# Tutorial 10: Multi-Sample Analysis

**Duration:** 25-30 minutes

This tutorial covers cohort-level analysis - extracting features from multiple samples and comparing them.

## Learning Objectives

By the end of this tutorial, you will be able to:
- Create and customize statistics panels
- Compute summary features for multiple samples
- Compare samples using dimensionality reduction
- Identify differentiating features between groups
- Export features for downstream ML

## Prerequisites

- Tutorials 1-9 completed
- Basic understanding of feature extraction concepts

## Biological Context

**Why multi-sample analysis?**
- Compare spatial organization across patients
- Identify biomarkers of treatment response
- Stratify patients by tissue microenvironment
- Build ML models for outcome prediction

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from spatialtissuepy import SpatialTissueData
from spatialtissuepy.summary import (
    StatisticsPanel,
    SpatialSummary,
    MultiSampleSummary,
    compute_summary,
    load_panel,
    list_metrics,
)
from spatialtissuepy.viz import (
    plot_spatial_scatter,
    plot_pca_samples,
    plot_metric_heatmap,
    plot_violin_comparison,
)

np.random.seed(42)

### Create synthetic cohort

We'll simulate a cohort of 12 samples:
- 6 "responders" (high immune infiltration)
- 6 "non-responders" (low immune infiltration)

In [ ]:
def create_sample(immune_level='high', sample_id='S1'):
    """Create a synthetic tissue sample."""
    # Tumor core
    n_tumor = np.random.randint(150, 250)
    tumor = np.random.normal(loc=[500, 500], scale=100, size=(n_tumor, 2))
    
    # Immune cells (more if responder)
    if immune_level == 'high':
        n_immune = np.random.randint(80, 150)
        immune_scale = 150  # More infiltration
    else:
        n_immune = np.random.randint(20, 60)
        immune_scale = 250  # More peripheral
    
    immune = np.random.normal(loc=[500, 500], scale=immune_scale, size=(n_immune, 2))
    
    # Stromal
    n_stromal = np.random.randint(50, 100)
    stromal = np.random.uniform(0, 1000, size=(n_stromal, 2))
    
    # Combine
    coords = np.vstack([tumor, immune, stromal])
    types = (['Tumor']*n_tumor + ['CD8_T_cell']*n_immune + ['Stromal']*n_stromal)
    
    return SpatialTissueData(
        coordinates=coords,
        cell_types=np.array(types)
    )

# Create cohort
samples = []
sample_ids = []
groups = []

for i in range(6):
    # Responders
    samples.append(create_sample('high', f'R{i+1}'))
    sample_ids.append(f'R{i+1}')
    groups.append('Responder')
    
    # Non-responders
    samples.append(create_sample('low', f'NR{i+1}'))
    sample_ids.append(f'NR{i+1}')
    groups.append('Non-Responder')

print(f"Created {len(samples)} samples")
print(f"  Responders: {sum(g == 'Responder' for g in groups)}")
print(f"  Non-Responders: {sum(g == 'Non-Responder' for g in groups)}")

In [ ]:
# Visualize example samples
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_spatial_scatter(samples[0], ax=axes[0])
axes[0].set_title(f'{sample_ids[0]} (Responder)')

plot_spatial_scatter(samples[1], ax=axes[1])
axes[1].set_title(f'{sample_ids[1]} (Non-Responder)')

plt.tight_layout()
plt.show()

## Section 1: Statistics Panels

### 1.1 Exploring Available Metrics

In [ ]:
# List all available metrics
metrics = list_metrics()

print(f"Available metrics: {len(metrics)}")
print("\nBy category:")

categories = {}
for m in metrics:
    cat = m.category
    if cat not in categories:
        categories[cat] = []
    categories[cat].append(m.name)

for cat, metric_list in categories.items():
    print(f"\n{cat}:")
    for m in metric_list[:5]:  # Show first 5
        print(f"  - {m}")
    if len(metric_list) > 5:
        print(f"  ... and {len(metric_list) - 5} more")

### 1.2 Creating Custom Panels

In [ ]:
# Create a custom panel
panel = StatisticsPanel(name='immune_analysis')

# Population metrics
panel.add('cell_counts')
panel.add('cell_proportions')

# Spatial statistics
panel.add('ripleys_h_max', max_radius=100)
panel.add('mean_nearest_neighbor_distance')

# Neighborhood metrics
panel.add('mean_neighborhood_entropy', radius=50.0)
panel.add('neighborhood_homogeneity', radius=50.0)

# Colocalization
panel.add('colocalization_score', type_a='CD8_T_cell', type_b='Tumor', radius=50.0)

print("Panel contents:")
for metric in panel.metrics:
    print(f"  - {metric.name}")

### 1.3 Using Predefined Panels

In [ ]:
# Load predefined panels
basic_panel = load_panel('basic')
spatial_panel = load_panel('spatial')

print("Basic panel metrics:")
for m in basic_panel.metrics:
    print(f"  - {m.name}")

## Section 2: Computing Summaries

### 2.1 Single Sample Summary

In [ ]:
# Compute summary for single sample
summary = SpatialSummary(samples[0], panel)

# View as dictionary
print("Summary as dict:")
results = summary.to_dict()
for key, value in list(results.items())[:10]:
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# View as pandas Series
series = summary.to_series()
print(f"\nSummary as Series (shape: {series.shape})")
print(series.head(10))

### 2.2 Multi-Sample Summary

In [ ]:
# Compute summaries for all samples
multi_summary = MultiSampleSummary(
    samples=samples,
    panel=panel,
    sample_ids=sample_ids
)

# Get as DataFrame
df = multi_summary.to_dataframe()

print(f"Feature matrix shape: {df.shape}")
print(f"Samples: {df.shape[0]}")
print(f"Features: {df.shape[1]}")
print(f"\nFeature columns:")
print(list(df.columns))

In [ ]:
# Add group labels
df['group'] = groups

print("\nFeature matrix preview:")
print(df.head())

## Section 3: Comparing Groups

### 3.1 Feature Comparison

In [ ]:
# Compare key features between groups
key_features = [
    'prop_CD8_T_cell',
    'coloc_CD8_T_cell_Tumor',
    'mean_neighborhood_entropy',
    'mean_homogeneity'
]

# Check which features exist
available_features = [f for f in key_features if f in df.columns]

print("Group Comparison:")
for feat in available_features:
    resp = df[df['group'] == 'Responder'][feat].mean()
    non_resp = df[df['group'] == 'Non-Responder'][feat].mean()
    print(f"\n{feat}:")
    print(f"  Responder: {resp:.4f}")
    print(f"  Non-Responder: {non_resp:.4f}")

In [ ]:
# Violin plots for key features
fig, axes = plt.subplots(1, len(available_features), figsize=(4*len(available_features), 5))
if len(available_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, available_features):
    # Simple boxplot
    data_resp = df[df['group'] == 'Responder'][feat]
    data_nonresp = df[df['group'] == 'Non-Responder'][feat]
    
    ax.boxplot([data_resp, data_nonresp], labels=['Resp', 'Non-Resp'])
    ax.scatter([1]*len(data_resp), data_resp, alpha=0.6)
    ax.scatter([2]*len(data_nonresp), data_nonresp, alpha=0.6)
    ax.set_ylabel(feat)
    ax.set_title(feat[:20] + '...' if len(feat) > 20 else feat)

plt.tight_layout()
plt.show()

### 3.2 Statistical Testing

In [ ]:
from scipy import stats

# Test for significant differences
print("Statistical Tests (Mann-Whitney U):")
print("="*50)

numeric_cols = df.select_dtypes(include=[np.number]).columns
p_values = {}

for col in numeric_cols:
    resp = df[df['group'] == 'Responder'][col].dropna()
    non_resp = df[df['group'] == 'Non-Responder'][col].dropna()
    
    if len(resp) > 2 and len(non_resp) > 2:
        stat, p = stats.mannwhitneyu(resp, non_resp, alternative='two-sided')
        p_values[col] = p

# Sort by p-value
sorted_pvals = sorted(p_values.items(), key=lambda x: x[1])

print("\nTop differentiating features:")
for feat, p in sorted_pvals[:10]:
    significance = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"  {feat}: p={p:.4f} {significance}")

## Section 4: Dimensionality Reduction

### 4.1 PCA Analysis

In [ ]:
# Prepare data for PCA
feature_cols = [c for c in df.columns if c not in ['group', 'sample_id']]
X = df[feature_cols].values

# Handle NaN values
X = np.nan_to_num(X, nan=0.0)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.1%}")

In [ ]:
# Plot PCA
fig, ax = plt.subplots(figsize=(8, 6))

colors = {'Responder': 'blue', 'Non-Responder': 'red'}

for group in ['Responder', 'Non-Responder']:
    mask = np.array(groups) == group
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=colors[group], label=group, s=100, alpha=0.7)

# Add sample labels
for i, sid in enumerate(sample_ids):
    ax.annotate(sid, (X_pca[i, 0], X_pca[i, 1]), fontsize=8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA of Spatial Features')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Feature Loadings

In [ ]:
# Top features driving PC1
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=['PC1', 'PC2']
)

print("Top features for PC1:")
print(loadings['PC1'].abs().nlargest(10))

## Section 5: Feature Heatmap

In [ ]:
# Create heatmap of features
fig, ax = plt.subplots(figsize=(12, 8))

# Select top features
top_features = list(loadings['PC1'].abs().nlargest(10).index)
heatmap_data = df[top_features].T
heatmap_data.columns = sample_ids

# Standardize for visualization
heatmap_data = (heatmap_data - heatmap_data.mean(axis=1).values[:, np.newaxis]) / heatmap_data.std(axis=1).values[:, np.newaxis]

im = ax.imshow(heatmap_data.values, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)

ax.set_xticks(range(len(sample_ids)))
ax.set_xticklabels(sample_ids, rotation=45, ha='right')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features)

plt.colorbar(im, label='Z-score')
ax.set_title('Feature Heatmap (Top PC1 Features)')

# Add group annotation
for i, g in enumerate(groups):
    color = 'blue' if g == 'Responder' else 'red'
    ax.axvline(i-0.5, color=color, linewidth=2, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6: Export for ML

In [ ]:
# Export feature matrix for downstream ML
import tempfile
import os

temp_dir = tempfile.mkdtemp()
output_path = os.path.join(temp_dir, 'spatial_features.csv')

df.to_csv(output_path, index=True)
print(f"Exported to: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nReady for sklearn, XGBoost, etc.")

## Exercise: Build a Classifier

1. Use the extracted features to build a simple classifier (e.g., logistic regression)
2. Evaluate with cross-validation
3. Identify the most important features for classification

In [ ]:
# Your code here


## Summary

In this tutorial, you learned:

- **Statistics panels:** Creating custom metric panels
- **Multi-sample summaries:** Computing features for cohorts
- **Group comparison:** Statistical testing between groups
- **Dimensionality reduction:** PCA for sample visualization
- **Feature export:** Preparing data for ML pipelines

**Key insights:**
- Spatial features can distinguish biological groups
- PCA reveals major axes of variation
- Feature heatmaps show sample clustering
- Exported features enable downstream ML

## Next Steps

- **Tutorial 11: PhysiCell Integration** - ABM analysis
- **Tutorial 12: Advanced Workflows** - Complete analysis pipelines